# Spoke gold layer

## Imports and constants

In [0]:
from pyspark.sql import functions as F

In [0]:
DATA_QUALITY_REPORT_TARGET = "spoke.gold.data_quality_report"
DAILY_STATION_USAGE_TARGET = "spoke.gold.daily_station_usage"
HOURLY_DEMAND_TARGET = "spoke.gold.hourly_demand"

## Read other layer's data

In [0]:
df_stations_silver = spark.table("spoke.silver.station")
df_rides_silver = spark.table("spoke.silver.rides")
df_rides_quarantine_silver = spark.table("spoke.silver.rides_quarantine")

## Forming data quality report

In [0]:
duplicate_ride_count = (df_rides_silver
                        .filter(F.col("identical_records") > 1)
                        .count())

missing_end_station_count = (df_rides_silver
                          .filter(F.col("is_unclosed") == True)
                          .count())

unknown_start_station = (df_rides_silver
                               .join(other=df_stations_silver,
                                     on=F.col("start_station_id") == F.col("station_id"),
                                     how="left_anti")
                               .select("ride_id"))

unknown_end_station = (df_rides_silver
                       .filter(F.col("end_station_id").isNotNull())
                       .join(other=df_stations_silver,
                             on=F.col("end_station_id") == F.col("station_id"),
                             how="left_anti")
                       .select("ride_id"))

unknown_station_count = (unknown_start_station
                         .union(unknown_end_station)
                         .distinct()
                         .count())

unreliable_duration_count = (df_rides_silver
                             .filter(F.col("has_wrong_duration") == True)
                             .count())

rows = [("duplicate_ride", duplicate_ride_count),
        ("missing_end_station", missing_end_station_count),
        ("unknown_station", unknown_station_count),
        ("unreliable_duration", unreliable_duration_count)]

df_data_quality_report = spark.createDataFrame(rows, schema="issue STRING, record_count BIGINT")

## Forming Daily station usage

### Generate date period

In [0]:
sequence = (F.explode(F.sequence("min_date", "max_date"))
            .alias("ride_date"))

df_dates = (df_rides_silver
                  .agg(F.min(F.to_date("started_at")).alias("min_date"),
                       F.max(F.to_date("ended_at")).alias("max_date"))
                  .select(sequence))

### Aggregate ride data

In [0]:
df_rides_from_station = (df_rides_silver
                         .filter(F.col("user_type") != "staff")
                         .groupBy(F.to_date("started_at").alias("ride_date_from"),
                                  "start_station_id")
                         .agg(F.count("ride_id").alias("rides_started")))

df_rides_to_station = (df_rides_silver
                         .filter(F.col("user_type") != "staff")
                         .groupBy(F.to_date("ended_at").alias("ride_date_to"),
                                  "end_station_id")
                         .agg(F.count("ride_id").alias("rides_ended")))

### Joining dataframes

In [0]:
dates_subset = ["ride_date"]
stations_subset = ["station_id", "station_name", "district"]
rides_subset = ["rides_started", "rides_ended"]

key_1 = ((F.col("start_station_id") == F.col("station_id"))
         & (F.col("ride_date") == F.col("ride_date_from")))

key_2 = ((F.col("end_station_id") == F.col("station_id"))
         & (F.col("ride_date") == F.col("ride_date_to")))

df_daily_station_usage = (df_dates
                          .crossJoin(df_stations_silver.select(stations_subset))
                          .join(df_rides_from_station,
                                on=key_1,
                                how="left")
                          .join(df_rides_to_station,
                                on=key_2,
                                how="left")
                          .dropna(how="all", subset=rides_subset)
                          .fillna(0, subset=rides_subset)
                          .withColumn("net_flow", F.col("rides_ended") - F.col("rides_started"))
                          .select(dates_subset + stations_subset + rides_subset + ["net_flow"])
                          .orderBy("ride_date", "station_id"))

## Forming Hourly demand

### Generate hourly period

In [0]:
df_hours = (spark.range(24)
            .select(F.col("id")
            .cast("int")
            .alias("hour_of_day")))

### Aggregate ride data

In [0]:
df_rides_per_hour = (df_rides_silver
                     .filter(F.col("user_type") != "staff")
                     .groupBy(F.hour("started_at").alias("ride_hour_from"))
                     .agg(F.count("ride_id").alias("rides_started"),
                          (F.avg("calculated_duration") / 60).alias("avg_ride_minutes")))

### Joining dataframes

In [0]:
rides_subset = ["rides_started", "avg_ride_minutes"]

df_hourly_demand = (df_hours
                    .join(df_rides_per_hour,
                          on=F.col("hour_of_day") == F.col("ride_hour_from"),
                          how="left")
                    .dropna(how="all", subset=rides_subset)
                    .fillna(0, subset=rides_subset)
                    .select(["hour_of_day"] + rides_subset))

## Write to delta tables

In [0]:
(df_data_quality_report.write
    .mode("overwrite")
    .saveAsTable(DATA_QUALITY_REPORT_TARGET))

In [0]:
(df_daily_station_usage.write
    .mode("overwrite")
    .saveAsTable(DAILY_STATION_USAGE_TARGET))

In [0]:
(df_hourly_demand.write
    .mode("overwrite")
    .saveAsTable(HOURLY_DEMAND_TARGET))